# Crypto News Impact Prediction with SageMaker XGBoost

Train model để dự đoán price impact từ news sentiment và technical indicators

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import boto3
import sagemaker
from sagemaker import get_execution_role
from sagemaker.inputs import TrainingInput
from sagemaker.xgboost import XGBoost
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# SageMaker session
session = sagemaker.Session()
role = get_execution_role()
bucket = 'crypto-news-datalake-training'

print(f"SageMaker role: {role}")
print(f"S3 bucket: {bucket}")

## 2. Load Data from S3

In [ ]:
# Load training data
s3_uri = f's3://{bucket}/training-data/training-data-manual-2026-01-28.csv'
df = pd.read_csv(s3_uri)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

## 3. Exploratory Data Analysis

In [ ]:
# Target distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.hist(df['returnPct'], bins=50, edgecolor='black')
plt.title('Return % Distribution')
plt.xlabel('Return %')

plt.subplot(1, 3, 2)
df['symbol'].value_counts().plot(kind='bar')
plt.title('Symbol Distribution')
plt.xticks(rotation=45)

plt.subplot(1, 3, 3)
df['timeframe'].value_counts().plot(kind='bar')
plt.title('Timeframe Distribution')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

print("\nTarget statistics:")
print(df['returnPct'].describe())

In [ ]:
# Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
# Define features
feature_cols = [
    # Sentiment
    'sentimentScore',
    'confidence',
    
    # Momentum
    'rsi14',
    'macdHistogram',
    
    # Trend
    'priceToSma200Ratio',
    'priceToEma50Ratio',
    
    # Volatility
    'volatilityAtr14',
    'bbWidth',
    
    # Liquidity
    'volumeRatio24h',
    
    # Market correlation
    'btcPriceChange24h',
    'btcDominance'
]

# Handle missing values
df_clean = df.dropna(subset=feature_cols + ['returnPct'])

print(f"Dataset after cleaning: {df_clean.shape}")
print(f"Rows dropped: {len(df) - len(df_clean)}")

# Prepare X and y
X = df_clean[feature_cols]
y = df_clean['returnPct']

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

## 5. Split Data

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Combine for XGBoost format (target in first column)
train_data = pd.concat([y_train, X_train], axis=1)
test_data = pd.concat([y_test, X_test], axis=1)

# Save to CSV
train_data.to_csv('train.csv', index=False, header=False)
test_data.to_csv('test.csv', index=False, header=False)

print("\nData saved to train.csv and test.csv")

## 6. Upload to S3

In [ ]:
# Upload training data to S3
train_s3_path = session.upload_data(
    path='train.csv',
    bucket=bucket,
    key_prefix='sagemaker/train'
)

test_s3_path = session.upload_data(
    path='test.csv',
    bucket=bucket,
    key_prefix='sagemaker/test'
)

print(f"Training data uploaded to: {train_s3_path}")
print(f"Test data uploaded to: {test_s3_path}")

## 7. Train XGBoost Model

In [ ]:
# Get XGBoost container
from sagemaker.image_uris import retrieve

container = retrieve('xgboost', session.boto_region_name, version='1.5-1')
print(f"XGBoost container: {container}")

# Create estimator
xgb_estimator = sagemaker.estimator.Estimator(
    container,
    role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{bucket}/sagemaker/output',
    sagemaker_session=session
)

# Set hyperparameters
xgb_estimator.set_hyperparameters(
    objective='reg:squarederror',
    num_round=100,
    max_depth=6,
    eta=0.1,
    gamma=4,
    min_child_weight=6,
    subsample=0.8,
    silent=0
)

print("\nEstimator configured")

In [ ]:
# Train model
xgb_estimator.fit({
    'train': TrainingInput(train_s3_path, content_type='text/csv'),
    'validation': TrainingInput(test_s3_path, content_type='text/csv')
})

print("\n✅ Training completed!")

## 8. Deploy Model

In [ ]:
# Deploy endpoint
predictor = xgb_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
    endpoint_name='crypto-news-impact-predictor'
)

print(f"\n✅ Model deployed to endpoint: {predictor.endpoint_name}")

## 9. Test Predictions

In [ ]:
# Make predictions
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

predictor.serializer = CSVSerializer()
predictor.deserializer = CSVDeserializer()

# Test with first 10 samples
test_sample = X_test[:10].values
predictions = predictor.predict(test_sample)

# Compare with actual
results_df = pd.DataFrame({
    'Actual': y_test[:10].values,
    'Predicted': [float(p[0]) for p in predictions]
})

results_df['Error'] = results_df['Actual'] - results_df['Predicted']
results_df['Error %'] = (results_df['Error'] / results_df['Actual'] * 100).round(2)

print(results_df)

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(results_df['Actual'], results_df['Predicted'], alpha=0.6)
plt.plot([results_df['Actual'].min(), results_df['Actual'].max()],
         [results_df['Actual'].min(), results_df['Actual'].max()],
         'r--', lw=2)
plt.xlabel('Actual Return %')
plt.ylabel('Predicted Return %')
plt.title('Actual vs Predicted Returns')
plt.grid(True, alpha=0.3)
plt.show()

## 10. Save Endpoint Info

In [ ]:
endpoint_info = {
    'endpoint_name': predictor.endpoint_name,
    'model_data': xgb_estimator.model_data,
    'features': feature_cols,
    'created_at': pd.Timestamp.now().isoformat()
}

import json
with open('endpoint_info.json', 'w') as f:
    json.dump(endpoint_info, f, indent=2)

print("Endpoint info saved to endpoint_info.json")
print(json.dumps(endpoint_info, indent=2))

## 🎯 Next Steps

1. Integrate endpoint với news-ai-service
2. Create prediction API: `POST /predict`
3. Monitor model performance
4. Retrain weekly với new data